#### Building an Excel File which includes all the UK Government Access Fuel Price Data

**This notebook has been built in Jupyter Notebook 7.4.7.**

**It using the dependencies: BeautifulSoup4, Pandas, Requests, Openpyxl**

If not already downloaded - install via terminal/cmd - or run github requirements.txt or run_script.py scripts - see README.MD in github for full directions


```
pip install pandas requests openpyxl beautifulsoup4
```


In [1]:
#CODE TO BUILD EXCEL FILE in a .xlsx format containing all the data listed for fuel prices provided at the
# UK government ACCESS FUEL PRICES website: https://www.gov.uk/guidance/access-fuel-price-data

#import the modules we need to build and export the DataFrame

from bs4 import BeautifulSoup #beautifulsoup 4 package is installed as part of Anaconda environment to parse HTML
import requests 
import pandas as pd
import json
from datetime import datetime
import os  # this will allow us to specify which path we want to save the final output excel file to


#### Scraping the URLs we need from the UK government website: 

In [2]:
# SCRAPE THE URLs:

# Define the URL of the webpage to scrape
url = 'https://www.gov.uk/guidance/access-fuel-price-data'

# use requests library to send GET request for the url

response = requests.get(url)

# Check if the request was successful - status should be 200 and parse with BeautifulSoup

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')

# Initialize a list to store JSON URLs
json_url = []


# The URLs are all held within table markdown tags. To find all <td> elements in the table, using soup:

td_elements = soup.find_all('td')

# Iterate over each <td> element and extract the URLs ending with .json or .html (shell data)
for td in td_elements:
    url = td.text.strip()
    if url.startswith('https://') and url.endswith('.json') or url.endswith('.html'):
        json_url.append(url)

In [ ]:
#CHECK json_url contains the various url json addresses

json_url

#### Building the intitial DataFrame

We need to use the following to generate a 'header' - which gives the appearance that the request is coming from a standard web browser. Otherwise the Tesco data will reject and timeout. We will then have to run another header sequence to get the Jet Local data - and merge that with all the other data - which will return a failed o fetch error.

In [ ]:

#building the dataframe

#headers - WITHOUT THIS - THE tesco_url ENTRY WILL REJECT THE REQUEST AND TIME OUT

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

#blank list to store the called json data

fuel_data = []

#loop through each URL and fetch the json data:
for url in json_url:
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        json_data = response.json()
        fuel_data.append(json_data)
    else:
        print(f"Failed to fetch data from {url}")
    
    
    
   
# Make DataFrame with pandas (pd) called df_fuel - and use the .json_normalize function get to record_path 'stations'

df_fuel = pd.json_normalize(fuel_data, record_path = 'stations')

In [ ]:
# returns the dataframe - for visual check 

df_fuel


### Fixing the missing Jet Local data

This is a header issue - but it's not being picked up by making the header statement above - which spoofs all the other url sites into thinking this is a normal user browser request. However - a modified headers statement can be used. We will use this to create a separate dataframe for the Jet json data - and then append to our other dataframe.

In [6]:
# Load json file from Jet URL:

url_jet = "https://jetlocal.co.uk/fuel_prices_data.json"  #The url for the jetlocal json data
headers = {"User-Agent": "Mozilla/5.0"}  # set a headers that will mimic a browser that works with this url

response = requests.get(url_jet, headers=headers)

if response.status_code == 200:
    json_data = response.json()
    df_jet = pd.DataFrame(json_data)
else:
    print(f"Request failed with status code {response.status_code}")


# place the json data in a dataframe - normalised to show the nested columns

df_jet = pd.json_normalize(json_data, record_path = 'stations')

In [ ]:
# test the jet dataframe works - call df_jet

df_jet

In [8]:
# append the df-jet dataframe to our df_fuel dataframe

df_combinded_fuel = pd.concat([df_fuel, df_jet], ignore_index=True)

In [ ]:
#check it's worked
df_combinded_fuel

In [10]:
#rename df_combined back to df_fuel

df_combined = df_fuel

#### Filtering our DataFrame for Scotland only postcode locations

The initial filter relies on using the two letter codes used in Sctoland - though some,could potentially also cover some England postcodes areas. This will cut the entries down substantially - allowing us to make use of an API to further check the postcodes locations

In [14]:

# Filter the DataFrame to include only postcodes starting with SCOTLAND postcode identifiers such as 'AB' or 'G'
# This will cut the new dataframe from 4554 entries down to about 500

df_poss_scot = df_fuel[df_fuel['postcode'].str.startswith(('AB', 'G', 'ML', 'DD', 'DG', 'HS', 'PA', 'PH', 'EH', 'IV', 'TD', 'FK', 'KA', 'KW', 'KY', 'ZE'))]



In [ ]:
# Check how many filtered datapoints for probable Scottish postcodes

len(df_poss_scot)

#### Checking our filtered list against a database - using an API call to postcodes.io database

**We first need to build a function that will allow us to make the api calls**

In [12]:
#We can now check to see which other postcodes are actually Scotland postcodes - calling the 500 through the postcodes api
# We build this function first - which calls the api.postcodes url - and then checks the json file and nested entry 'country'
# if there's a match for Scotland - it returns a True value - otherwise - it returns False

# Function to check if a postcode is in Scotland

def is_postcode_in_scotland(postcode):
    url = f"http://api.postcodes.io/postcodes/{postcode}"
    response = requests.get(url)
    if response.status_code == 200:
        result = response.json()['result']
        if result and result['country'] == 'Scotland':
            return True
    return False


#### Building an updated dataframe with a new columns 'Wales' which will list whether the postcode is in Wales (True) or not (False)

In [ ]:
# Update the 'Wales' column based on the postcode check 
# this applies the function we've just built - adds a new column 'Wales' to the dataframe df_poss_wales - which will be either True or False



df_scotland = df_poss_scot

df_scotland['Scotland'] = df_scotland['postcode'].apply(is_postcode_in_scotland)

**While a warning is thrown up - it does actually create the correct dataframe, listing a column of True and False values for Scotland**

#### We can now filter the new dataframe to remove all False entries

In [15]:
#removes all false enteries - leaving only Wales enteries

df_scotland = df_scotland[df_scotland.Scotland != False]

In [16]:
# THIS ALLOWS US TO SCROLL THROUGH THE ENTIRE DATA FRAME - it's only 300 rows - UNCOMMENT IF YOU WANT TO DISPLAY WHOLE DATAFRAME

# pd.set_option('display.max_rows', None)

In [ ]:
df_scotland


In [ ]:
#show length of dataframe in rows

len(df_scotland)

#### Data Validation and cleaning

In [25]:
#From looking at the data - potential duplicates exist - especially for PRL branded as both PRL and MFG
# we use the .duplicated method in Pandas - stating which column to look at - in our case 'postcode'
# The keep = False retains both sets of duplicates - so they can be checked physically to ensure they are true duplicates

duplicates_scotland = df_scotland[df_scotland.duplicated(['postcode'], keep = False)]

In [ ]:
# displays the dupicates - some are not true duplicates and need checking

duplicates_scotland

In [26]:
# YOU CAN USE THIS TO REMOVE TRUE DUPLICATES - IF THEY EXIST - OTHERWISE - COMMENT OUT
# uses the .drop method to remove multiple index items - these are the duplicate index numbers. Note - the index numbers can change 
# which means you will have to manually check them and add them to the list below

df_scotland_cleaner = df_scotland.drop(index =[#add in duplicate index items eg: 2278, 2629 here])

In [29]:
len(df_scotland_cleaner)

369

#### RENAME the cleaned dataframe to df_scotland

In [31]:
df_scotland_cleaner = df_scotland

#### We now have a clean and validated DataFrame of all Scotland only fuel stations reporting to the UK Access Fuel site

#### One final bit of housekeeping - in case we want to use the Latitude and Longitdue locations later
**Rename them first - and then we want to change the figures from a string object to a real float number**

In [32]:
#rename lat and longitude columns to single words

df_scotland_lat = df_scotland.rename(columns={'location.latitude' : 'latitude', 'location.longitude' : 'longitude'})

In [33]:
#convert lat and long from string object to float numbers

df_scotland_lat['latitude'] = df_scotland_lat.latitude.astype(float)
df_scotland_lat['longitude'] = df_scotland_lat.longitude.astype(float)

#### Exporting the final DataFrame to an Excel file format - in this case stored on a hard-drive volume

In [42]:
# export to file

# use datetime to append date to file name


current_date = datetime.now().strftime('%d-%m-%Y')

output = f'fuel_prices_scotland_{current_date}.xlsx'

In [ ]:
# Specify the path to save the file 


#Requests user to add required path location:

path_to_save = input("Please type the path location to save your file, for example C:\Data or /Volumes/portable_drive/data (DO NOT ADD FILENAME")

path_to_save_expanded = os.path.expanduser(path_to_save)

# Full file path
file_path = os.path.join(path_to_save_expanded, output)

In [44]:
#convert fuel DataFrame data to Excel format and store in directory specified above with date appended

df_scotland_lat.to_excel(file_path, index = False)

IF YOU WANT TO OUTPUT THE ENTIRE DATASET FOR ALL UK DATA - UNCOMMENT BELOW AND RUN

In [31]:
#Output UK data to excel
#df_fuel.to_excel(file_path, index = False)